# Étape 3 — Classification ABC × XYZ × K-Means

**Objectif** : segmenter les 250 produits en classes homogènes pour appliquer ensuite une politique de gestion différenciée (modèle de prévision, seuil de réapprovisionnement, etc.).

**Méthodologie (mémoire §3.4)** :
1. **ABC** sur le CA cumulé 36 mois — seuils 70 % / 90 %.
2. **XYZ** sur le coefficient de variation des ventes mensuelles — seuils 0.5 / 1.0, repli sur quantiles 33/66 si X vide.
3. **Matrice 3×3 ABC × XYZ** — nb produits + part du CA.
4. **K-Means** : log-transform des features asymétriques, StandardScaler, k testé de 2 à 10, choix par silhouette (k ≥ 3), interprétation métier.
5. **Projection PCA 2D** pour visualisation.

**Entrée** : `data/features/product_features.csv` (Étape 2).

**Sorties** :
- `outputs/tables/classification_produits.csv` (`produit_id, classe_abc, classe_xyz, cluster_kmeans, libelle_cluster, ...`).
- `outputs/tables/abc_xyz_matrix.csv`, `kmeans_diagnostics.csv`, `cluster_profile.csv`.
- `outputs/figures/cls_01..07.png` (7 figures).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')

from src.classification import classify_pipeline, abc_xyz_matrix
from src.utils import FEATURES_DIR

## 1. Chargement et lancement de la classification

In [ ]:
product_feats = pd.read_csv(FEATURES_DIR/'product_features.csv')
final, diag = classify_pipeline(product_feats)
print(f"Produits classés : {len(final)}")
print(f"k* retenu : {diag.best_k} (silhouette {diag.best_silhouette:.3f})")

## 2. Classification ABC — courbe de Pareto

In [ ]:
final['classe_abc'].value_counts().reindex(['A','B','C'])

In [ ]:
s = final.sort_values('ca_total_36mois', ascending=False).reset_index(drop=True)
total = s['ca_total_36mois'].sum()
cum = s['ca_total_36mois'].cumsum()/total*100
colors = s['classe_abc'].map({'A':'#1f4e79','B':'#ff9f1c','C':'#6c757d'})
fig, ax = plt.subplots(figsize=(13,6))
ax.bar(range(len(s)), s['ca_total_36mois'], color=colors, edgecolor='black', linewidth=0.2)
ax2 = ax.twinx()
ax2.plot(range(len(s)), cum, color='black', linewidth=2)
ax2.axhline(70, color='green', ls='--', label='Seuil A (70%)')
ax2.axhline(90, color='orange', ls='--', label='Seuil B (90%)')
ax2.set_ylim(0,105); ax2.set_ylabel('CA cumulé (%)')
ax.set_title('Courbe de Pareto — Classification ABC')
ax.set_xlabel('Produits triés par CA décroissant'); ax.set_ylabel('CA produit (USD)')
ax2.legend()
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/cls_01_pareto_abc.png', dpi=120, bbox_inches='tight'); plt.show()

**Lecture** : la courbe noire (CA cumulé) franchit le seuil 70 % au bout de 69 produits, et 90 % au bout de 128. La classe A pèse donc lourd (1/3 des références), ce qui est typique d'un catalogue informatique où certaines marques (HP, Dell, Cisco, Samsung) génèrent une forte concentration de revenus.

## 3. Classification XYZ

In [ ]:
final['classe_xyz'].value_counts().reindex(['X','Y','Z'])

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
counts = final['classe_xyz'].value_counts().reindex(['X','Y','Z']).fillna(0).astype(int)
sns.barplot(x=counts.index, y=counts.values, hue=counts.index,
            palette={'X':'#2d6a4f','Y':'#ff9f1c','Z':'#ef476f'}, ax=ax, legend=False)
for x,y in zip(counts.index, counts.values):
    ax.text(x, y, str(int(y)), ha='center', va='bottom')
ax.set_title('Répartition XYZ — variabilité de la demande')
ax.set_ylabel('Nombre de produits'); ax.set_xlabel('Classe XYZ')
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/cls_02_distribution_xyz.png', dpi=120, bbox_inches='tight'); plt.show()

## 4. Matrice croisée ABC × XYZ

In [ ]:
abc_xyz_matrix(final)

In [ ]:
nb = final.groupby(['classe_abc','classe_xyz']).size().unstack(fill_value=0).reindex(index=['A','B','C'], columns=['X','Y','Z'], fill_value=0)
ca = final.groupby(['classe_abc','classe_xyz'])['ca_total_36mois'].sum().unstack(fill_value=0).reindex(index=['A','B','C'], columns=['X','Y','Z'], fill_value=0)
ca_pct = (ca/ca.values.sum()*100).round(1)
fig, axes = plt.subplots(1,2, figsize=(13,5))
sns.heatmap(nb, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar_kws={'label':'Produits'})
axes[0].set_title('Nombre de produits')
sns.heatmap(ca_pct, annot=True, fmt='.1f', cmap='OrRd', ax=axes[1], cbar_kws={'label':'% du CA total'})
axes[1].set_title('Part du CA (%)')
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/cls_03_matrice_abc_xyz.png', dpi=120, bbox_inches='tight'); plt.show()

**Lecture stratégique** :
- **AX et AY** (66 produits) génèrent ≈ 67 % du CA — priorité absolue, prévisions fiables possibles.
- **CY et CZ** (69 produits) ne pèsent que ≈ 6 % du CA — gestion plus simple, modèles statistiques ou Croston.
- **AZ** (3 produits A à demande irrégulière) demande un traitement spécifique : forte valeur + volatilité élevée.

## 5. K-Means — méthode du coude et silhouette

In [ ]:
pd.DataFrame({'k': list(diag.inertia.keys()), 'inertia': list(diag.inertia.values()), 'silhouette': list(diag.silhouette.values())})

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,5))
ks = sorted(diag.inertia.keys())
axes[0].plot(ks, [diag.inertia[k] for k in ks], marker='o', linewidth=2, color='#1f4e79')
axes[0].axvline(diag.best_k, ls='--', color='red', label=f'k* = {diag.best_k}')
axes[0].set_title('Méthode du coude — inertie intra-cluster'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertie'); axes[0].legend()
axes[1].plot(ks, [diag.silhouette[k] for k in ks], marker='o', linewidth=2, color='#ff6b6b')
axes[1].axvline(diag.best_k, ls='--', color='red', label=f'k* = {diag.best_k}')
axes[1].set_title('Score de silhouette'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette moyenne'); axes[1].legend()
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/cls_04_elbow_silhouette.png', dpi=120, bbox_inches='tight'); plt.show()

## 6. Profil métier des clusters

In [ ]:
diag.cluster_profile

In [ ]:
final['libelle_cluster'].value_counts()

## 7. Projection PCA 2D

In [ ]:
fig, ax = plt.subplots(figsize=(11,7))
palette = sns.color_palette('tab10', n_colors=diag.best_k)
for cid in sorted(final['cluster_kmeans'].unique()):
    mask = final['cluster_kmeans']==cid
    label = final.loc[mask,'libelle_cluster'].iloc[0]
    ax.scatter(diag.pca_components[mask.values,0], diag.pca_components[mask.values,1],
               color=palette[int(cid)%len(palette)], label=f'{int(cid)} — {label}',
               alpha=0.7, s=50, edgecolors='black', linewidth=0.3)
ax.set_xlabel('Composante principale 1'); ax.set_ylabel('Composante principale 2')
ax.set_title('Projection PCA 2D des clusters K-Means')
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/cls_05_pca_2d.png', dpi=120, bbox_inches='tight'); plt.show()

## 8. Conclusion

| Classe | Produits | Part du CA | Stratégie prévision (Étape 5) |
|---|---:|---:|---|
| AX | 20 | 21 % | LSTM (haut enjeu, demande stable) |
| AY | 46 | 47 % | LSTM (haut enjeu, demande modérément variable) |
| AZ | 3  | 2 % | Croston (haut enjeu, demande très irrégulière) |
| BX/BY/BZ | 59 | 20 % | LightGBM |
| CX/CY | 104 | 9 % | SARIMA |
| CZ | 18 | 1 % | Croston |

**Clusters K-Means** : 3 groupes lisibles — *Bestseller stable* (126 produits à forte rotation à valeur élevée), *Rotation modérée* (88, consommables fréquents bon marché), *Dormant / risque obsolescence* (36, à surveiller à l'Étape 4).